In [4]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

# === Setup Mediapipe Pose ===
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, 
                    min_detection_confidence=0.5, 
                    min_tracking_confidence=0.5)

# === Helper: calculate angle between 3 points ===
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    return angle

# === Input video ===
video_name = "squat_good.mp4"   # <-- change to your video file
video_path = f"../data/{video_name}"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise FileNotFoundError(f"❌ Cannot open video file: {video_path}")

# === Storage ===
all_keypoints = []
angles_data = []

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark

        # --- Save raw keypoints ---
        keypoints = []
        for lm in landmarks:
            keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])
        all_keypoints.append([frame_id] + keypoints)

        # --- Extract important joints ---
        # Right side
        shoulder_r = [landmarks[12].x, landmarks[12].y]
        elbow_r    = [landmarks[14].x, landmarks[14].y]
        wrist_r    = [landmarks[16].x, landmarks[16].y]
        hip_r      = [landmarks[24].x, landmarks[24].y]
        knee_r     = [landmarks[26].x, landmarks[26].y]
        ankle_r    = [landmarks[28].x, landmarks[28].y]

        # Left side
        shoulder_l = [landmarks[11].x, landmarks[11].y]
        elbow_l    = [landmarks[13].x, landmarks[13].y]
        wrist_l    = [landmarks[15].x, landmarks[15].y]
        hip_l      = [landmarks[23].x, landmarks[23].y]
        knee_l     = [landmarks[25].x, landmarks[25].y]
        ankle_l    = [landmarks[27].x, landmarks[27].y]

        # --- Calculate angles ---
        knee_angle_r      = calculate_angle(hip_r, knee_r, ankle_r)
        knee_angle_l      = calculate_angle(hip_l, knee_l, ankle_l)
        elbow_angle_r     = calculate_angle(shoulder_r, elbow_r, wrist_r)
        elbow_angle_l     = calculate_angle(shoulder_l, elbow_l, wrist_l)
        hip_angle_r       = calculate_angle(shoulder_r, hip_r, knee_r)
        hip_angle_l       = calculate_angle(shoulder_l, hip_l, knee_l)
        shoulder_angle_r  = calculate_angle(elbow_r, shoulder_r, hip_r)
        shoulder_angle_l  = calculate_angle(elbow_l, shoulder_l, hip_l)

        # Save frame results
        angles_data.append([
            frame_id,
            knee_angle_r, knee_angle_l,
            elbow_angle_r, elbow_angle_l,
            hip_angle_r, hip_angle_l,
            shoulder_angle_r, shoulder_angle_l
        ])

    frame_id += 1

cap.release()

# === Save raw keypoints ===
keypoints_cols = ["frame_id"]
for i in range(33):  # 33 landmarks
    keypoints_cols += [f"x_{i}", f"y_{i}", f"z_{i}", f"v_{i}"]

df_keypoints = pd.DataFrame(all_keypoints, columns=keypoints_cols)
keypoints_file = f"../data/{video_name.split('.')[0]}_keypoints.csv"
df_keypoints.to_csv(keypoints_file, index=False)
print(f"✅ Raw keypoints saved to {keypoints_file} ({df_keypoints.shape})")

# === Save angles ===
df_angles = pd.DataFrame(angles_data, columns=[
    "frame_id",
    "knee_angle_r", "knee_angle_l",
    "elbow_angle_r", "elbow_angle_l",
    "hip_angle_r", "hip_angle_l",
    "shoulder_angle_r", "shoulder_angle_l"
])
angles_file = f"../data/{video_name.split('.')[0]}_angles.csv"
df_angles.to_csv(angles_file, index=False)
print(f"✅ Angles saved to {angles_file} ({df_angles.shape})")

df_angles.head()

I0000 00:00:1756721544.836851 1233506 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1756721544.908576 1240564 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1756721544.923142 1240564 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✅ Raw keypoints saved to ../data/squat_good_keypoints.csv ((202, 133))
✅ Angles saved to ../data/squat_good_angles.csv ((202, 9))


,frame_id,knee_angle_r,knee_angle_l,elbow_angle_r,elbow_angle_l,hip_angle_r,hip_angle_l,shoulder_angle_r,shoulder_angle_l
0,0,177.394652,168.808461,30.354890,147.934769,159.076229,149.086017,78.546024,103.415284
1,1,177.686264,168.560494,33.232018,149.141822,158.927480,148.645605,79.058880,103.890653
2,2,177.927484,168.918439,31.346591,149.909764,158.853347,148.973143,74.741875,103.611695
3,3,177.716090,169.023143,137.046081,150.698005,158.355034,149.200827,96.039244,103.926797
4,4,176.990261,169.812086,133.599620,151.804409,157.403248,149.287709,95.013640,103.901466
